In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)

In [3]:
train_data = pd.read_parquet("data/train_data_engineered.parquet")
test_data = pd.read_parquet("data/test_data_engineered.parquet")

train_data["is_night"] = train_data["is_night"].astype(int)
test_data["is_night"] = test_data["is_night"].astype(int)

X_train = train_data.drop(columns="is_fraud")
y_train = train_data["is_fraud"]
X_test = test_data.drop(columns="is_fraud")
y_test = test_data["is_fraud"]

print(X_train.shape, X_test.shape)

(1296675, 24) (555719, 24)


In [ ]:
neg, pos = y_train.value_counts()[0], y_train.value_counts()[1]
scale_pos_weight = neg / pos

print(f"legit: {neg}, fraud: {pos}, scale_pos_weight: {scale_pos_weight:.2f}")

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)